## Project Context: Intelligent Invoice Parsing using Databricks AI

In many real-world teams, invoices and receipts arrive as PDFs or scanned images, each following a different layout and vendor format.
While humans can visually understand these documents easily, extracting structured information from them at scale is slow, manual, and error-prone.

The goal of this project is to build an **intelligent document parsing pipeline** that can:
- Accept raw invoices in PDF or image format
- Automatically understand document layout and content
- Extract key business fields such as invoice number, subtotal, tax, and total
- Convert unstructured documents into structured, analytics-ready tables

This notebook demonstrates how Databricks’ `ai_parse_document` capability can be combined with Spark-based data engineering to achieve this in a scalable and production-oriented way.


In [0]:
#location for raw invoice PDFs.
base_path = "/Volumes/greensheen_catalog/default/raw_invoices"

In [0]:
#visually inspect PDF names and sizes
files = dbutils.fs.ls(base_path)
display(files)

## Reading Raw Documents (PDFs)

Invoices and receipts are not plain text files they are binary documents.
To preserve their original structure and visual layout, we read them as binary data instead of attempting text extraction at this stage.

At this point, we are not interpreting or extracting meaning.
We are simply loading the raw documents into the system so they can later be processed by an AI-powered document parser.

In [0]:
## Ingest raw invoice PDFs as binary files.
## raw binary content (required for ai_parse_document),

raw_df = (
  spark.read.format("binaryFile")
  .option("pathGlobFilter", "*.pdf")
  .load(base_path)
  .select("path", "content", "length", "modificationTime")
)

display(raw_df)

## Intelligent Document Parsing using ai_parse_document

This is the core step of the project.

The `ai_parse_document` function performs more than traditional OCR:
- It reads text from scanned or digital documents
- It understands document layout (pages, blocks, tables)
- It produces a structured representation of the document content

Instead of returning plain text, this step outputs a rich, nested structure describing **what the document contains and how it is organized**.

This structured representation becomes the foundation for all downstream extraction and business logic.


In [0]:

# Parse each raw PDF using Databricks ai_parse_document.
# converting raw binary content into a structured document representation
#The output is stored as a VARIANT column for downstream extraction and validation.

from pyspark.sql.functions import expr

parsed_df = raw_df.select(
    "path",
    expr("ai_parse_document(content) as parsed_document")
)


In [0]:
display(parsed_df.limit(2))

## Extracting Document Elements

The parsed document output contains a nested collection of elements, such as:
- Text blocks
- Table cells
- Headers and footers
- Page-level metadata

To make this information usable, we extract the document elements field and prepare it for flattening.


In [0]:
## ai_parse_document returns a nested VARIANT structure, so we first extract the elements array as a JSON string.
# This prepares the data for explicit schema definition and row-level 

elements_json_df = parsed_df.select(
    "path",
    expr("cast(variant_get(parsed_document, '$.document.elements') as string) as elements_json")
)

display(elements_json_df)



In [0]:
# STEP 1: Define the structure of a single document element produced by ai_parse_document.
# Each element represents a small piece of the document such as:
# - a line of text
# - a table cell
# - a header or footer
# along with its page number and position (bounding box) in the document.

# STEP 2: Convert the raw JSON string of document elements into a typed Spark structure
# using the schema defined above.


from pyspark.sql.functions import col, from_json, explode

elements_schema = """
array<struct<
  id: int,
  type: string,
  content: string,
  page_id: int,
  bbox: struct<coord: array<int>, page_id: int>,
  description: string
>>
"""

# Explode the elements array so that:
# - each row corresponds to ONE document element
# - a single PDF becomes many rows (one per element)

# After this step, we can easily filter, search, and reason over individual
# parts of the invoice instead of treating the document as a single blob.

elements_df = (
    elements_json_df
    .select("path", explode(from_json(col("elements_json"), elements_schema)).alias("e"))
    .select(
        "path",
        col("e.id").alias("element_id"),
        col("e.type").alias("element_type"),
        col("e.content").alias("content"),
        col("e.page_id").alias("page_id"),
        col("e.bbox.coord").alias("bbox_coord"),
        col("e.description").alias("description")
    )
)

display(elements_df)


In [0]:
# Identify candidate document elements that may contain invoice-level monetary fields.
# We perform a case-insensitive keyword scan over parsed document content to surface elements related to totals, taxes, shipping, and balances.

from pyspark.sql.functions import lower

keywords = ["total", "subtotal", "tax", "gst", "vat", "shipping", "freight", "discount", "amount due", "balance due"]

invoice_amount_candidates = (
    elements_df
    .withColumn("content_l", lower(col("content")))
    .where(" OR ".join([f"content_l LIKE '%{k}%'" for k in keywords]))
    .select("path", "page_id", "element_type", "content", "bbox_coord")
    .orderBy("path", "page_id", "element_id")
)

display(invoice_amount_candidates)



## Extracting Monetary Fields from Candidate Elements

After identifying candidate elements, we clean and normalize the extracted text to remove HTML artifacts, extra whitespace, and casing inconsistencies.

We then extract invoice-level monetary fields (subtotal, shipping, tax, total) using label-based patterns.
This step is intentionally tolerant and works even when multiple fields appear in the same table row.

Finally, we reshape the extracted values into a canonical `(field, amount)` format, making the data easier to validate, aggregate, and analyze downstream.


In [0]:
from pyspark.sql import functions as F

# 1) Clean HTML -> plain text : Clean the extracted document text, The parsed document content may contain HTML tags, extra spaces, and mixed casing.
pairs_src = (
    invoice_amount_candidates
    .withColumn("plain", F.regexp_replace(F.col("content"), r"<[^>]+>", " "))
    .withColumn("plain", F.regexp_replace(F.col("plain"), r"\s+", " "))
    .withColumn("plain_l", F.lower(F.col("plain")))
)

# 2) Extract each field from the same row (works even if all fields are in one table row)
invoice_amounts = (
    pairs_src
    .select(
        "path",
        F.regexp_extract("plain", r"Subtotal:\s*\$?\s*(\d[\d,]*\.?\d{0,2})", 1).alias("subtotal"),
        F.regexp_extract("plain", r"Shipping:\s*\$?\s*(\d[\d,]*\.?\d{0,2})", 1).alias("shipping"),
        F.regexp_extract("plain", r"Tax[^:]*:\s*\$?\s*(\d[\d,]*\.?\d{0,2})", 1).alias("tax"),
        F.regexp_extract("plain", r"(Total Due:|Amount Due:|Balance Due:)\s*\$?\s*(\d[\d,]*\.?\d{0,2})", 2).alias("total")
    )
    # 3) Convert wide columns into a long (field, amount) format
    # We reshape the data so that:
            # - each row represents ONE extracted amount
            # - columns become: (invoice path, field name, numeric amount)
    .selectExpr(
        "path",
        "stack(4, 'subtotal', subtotal, 'shipping', shipping, 'tax', tax, 'total', total) as (field, amount_str)"
    )
    .withColumn("amount", F.expr("try_cast(regexp_replace(amount_str, '[, ]', '') as double)")) #Clean extracted amounts
    .where(F.col("amount").isNotNull())
    .select("path", "field", "amount")
)

display(invoice_amounts)


## Aggregating Extracted Fields to Invoice Level

Earlier steps produced multiple rows per invoice, as each document was decomposed into individual elements.
Here, we reconstruct a **one-row-per-invoice** view.

We first create a complete list of invoice paths, then pivot extracted `(field, amount)` values back into columns
(`subtotal`, `shipping`, `tax`, `total`).

A left join ensures that invoices with missing or partially extracted fields are still included,
making gaps explicit rather than silently dropping data.


In [0]:
## Create a one-row-per-invoice list -Earlier, each invoice was split into many rows (one per document element).
# Here, we deduplicate paths so every invoice is tracked in the final output.

invoice_files = (
    elements_df
    .select("path")
    .distinct()
)
display(invoice_files)


In [0]:
# Reshape extracted amounts back to one row per invoice.
# We pivot (field -> columns) to create subtotal, shipping, tax, and total columns,
# then left join with the full invoice list so invoices with missing values are still included for validation and reporting.


from pyspark.sql import functions as F

all_paths = invoice_files.select("path").distinct()

pivot_financials = (
    invoice_amounts
    .groupBy("path")
    .pivot("field", ["subtotal", "shipping", "tax", "total"])
    .agg(F.max("amount"))
)

invoice_financials = (
    all_paths
    .join(pivot_financials, on="path", how="left")
)

display(invoice_financials)

## Validating Extracted Totals and Flagging Issues

Extraction alone is not enough, we also need trust signals.

Here we compute a `calculated_total = subtotal + shipping + tax` (treating missing values as 0),
compare it with the extracted `total`, and record the absolute difference.

We then assign a simple `status` label to make issues explicit (missing fields or total mismatch),
so the final output is not just extracted data, but **validated, debuggable invoice-level results**.


In [0]:
invoice_gold = (
    invoice_financials
    .withColumn(
        "calculated_total",
        F.coalesce(F.col("subtotal"), F.lit(0.0)) +
        F.coalesce(F.col("shipping"), F.lit(0.0)) +
        F.coalesce(F.col("tax"), F.lit(0.0))
    )
    .withColumn("diff", F.abs(F.col("total") - F.col("calculated_total")))
    .withColumn("is_consistent", F.col("diff") <= F.lit(0.05))
    .withColumn(
        "status",
        F.when(F.col("total").isNull(), "MISSING_TOTAL")
         .when(F.col("subtotal").isNull(), "MISSING_SUBTOTAL")
         .when(F.col("shipping").isNull(), "MISSING_SHIPPING")
         .when(F.col("tax").isNull(), "MISSING_TAX")
         .when(~F.col("is_consistent"), "TOTAL_MISMATCH")
         .otherwise("OK")
    )
)

display(invoice_gold)

## the Final Gold Table

After extraction, aggregation, and validation, we persist the final invoice-level dataset as a managed table.

This `invoice_gold` table represents the **clean, validated, one-row-per-invoice output** of the pipeline
and can now be used directly for analytics, reporting, or downstream AI applications.


In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS greensheen_catalog.gs_invoices")

invoice_gold.write.mode("overwrite").saveAsTable("greensheen_catalog.gs_invoices.invoice_gold")
